# External Validation on YBT Dataset

This notebook loads the saved C4 training artifacts (feature schema, scaler, models), prepares the YBT dataset to match the training feature space, applies the trained models, and reports performance and outputs predictions.

**Key Points:**
- Leakage-free: uses the saved scaler (fitted on C4 train) and does not refit on YBT
- Strict feature alignment: uses the exact feature names and order from training (45 features)
- Robust preprocessing: replicates questionnaire scoring and key feature engineering used in C4
- **SPQ missing**: YBT does not have SPQ (Sensory Perception Quotient) - all SPQ features filled with 0
- **AQ excluded**: AQ features are excluded from final feature set to prevent data leakage
- **SQR = Systemizing Quotient-Revised** (not Social Responsiveness Scale)


In [ ]:
# Imports and config
import os
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score, roc_curve

# Paths
ARTIFACT_DIR = '/Users/eb2007/playground/bullpy/c4_play2/models/cross_validation'
FEATURE_INFO_PATH = os.path.join(ARTIFACT_DIR, 'feature_info_original.json')
SCALER_PATH = os.path.join(ARTIFACT_DIR, 'scaler_original.joblib')
MODELS = {
    'Logistic Regression': os.path.join(ARTIFACT_DIR, 'logistic_regression_original.joblib'),
    'Random Forest': os.path.join(ARTIFACT_DIR, 'random_forest_original.joblib'),
    'XGBoost': os.path.join(ARTIFACT_DIR, 'xgboost_original.joblib'),
    'LightGBM': os.path.join(ARTIFACT_DIR, 'lightgbm_original.joblib'),
    'Gradient Boosting': os.path.join(ARTIFACT_DIR, 'gradient_boosting_original.joblib'),
}

# YBT dataset path
DATA_PATH = '/Users/eb2007/Library/CloudStorage/OneDrive-UniversityofCambridge/Documents/PhD/data/YBT.csv'

np.random.seed(42)


In [ ]:
# Load training artifacts (feature schema, scaler, models)
with open(FEATURE_INFO_PATH, 'r') as f:
    feature_info = json.load(f)

feature_names = feature_info['feature_names']
excluded_features = set(feature_info.get('excluded_features', []))
print(f"Loaded feature schema with {len(feature_names)} features")
print(f"Excluded features (AQ-related, will not be used): {list(excluded_features)[:5]}...")

scaler = joblib.load(SCALER_PATH)
print("Loaded saved StandardScaler (trained on C4)")

loaded_models = {}
for name, path in MODELS.items():
    if os.path.exists(path):
        loaded_models[name] = joblib.load(path)
print(f"Loaded {len(loaded_models)} trained models: {list(loaded_models.keys())}")


## STEP 1: Load YBT Data and Inspect Structure


In [ ]:
# Load YBT data
print(f"Loading YBT data from: {DATA_PATH}")
df = pd.read_csv(DATA_PATH)

print(f"\nYBT dataset shape: {df.shape}")
print(f"Columns: {len(df.columns)}")

# Verify questionnaire columns exist
print("\n" + "="*80)
print("CHECKING QUESTIONNAIRE COLUMNS")
print("="*80)

eq_cols = [col for col in df.columns if col.startswith('eq10_')]
sqr_cols = [col for col in df.columns if col.startswith('sq10_')]
aq_cols = [col for col in df.columns if col.startswith('aq_') and len(col) <= 5]  # aq_1 to aq_10

print(f"\nEQ columns found: {len(eq_cols)} - {sorted(eq_cols)}")
print(f"SQR columns found: {len(sqr_cols)} - {sorted(sqr_cols)}")
print(f"AQ columns found: {len(aq_cols)} - {sorted(aq_cols)}")

# Check for SPQ (should not exist in YBT)
spq_cols = [col for col in df.columns if col.startswith('spq_')]
if spq_cols:
    print(f"\n⚠️  WARNING: Found SPQ columns in YBT: {spq_cols}")
    print("   This is unexpected - YBT should not have SPQ")
else:
    print(f"\n✅ Confirmed: No SPQ columns in YBT (as expected)")

# Check demographics
demographic_cols = ['age', 'sex', 'gender']
print(f"\nDemographic columns:")
for col in demographic_cols:
    if col in df.columns:
        print(f"  ✅ {col}: {df[col].dtype}")
    else:
        print(f"  ❌ {col}: NOT FOUND")

# Check diagnosis columns
diagnosis_cols = [col for col in df.columns if 'diagnosis' in col.lower()]
print(f"\nDiagnosis columns found: {diagnosis_cols}")


## STEP 2: Convert Text Responses to Numeric

YBT uses text responses (e.g., "strongly agree", "slightly disagree") that need to be converted to numeric values.


In [ ]:
# YBT uses text responses, need to convert to numeric
print("="*80)
print("CONVERTING TEXT RESPONSES TO NUMERIC")
print("="*80)

response_mapping = {
    'strongly agree': 4,
    'slightly agree': 3,
    'slightly disagree': 2,
    'strongly disagree': 1
}

print(f"Response mapping: {response_mapping}")

# Convert all questionnaire columns
all_q_cols = eq_cols + sqr_cols + aq_cols
print(f"\nConverting {len(all_q_cols)} questionnaire columns...")

for col in all_q_cols:
    if col in df.columns:
        # Convert to string, strip whitespace, lowercase, then map
        df[col] = df[col].astype(str).str.strip().str.lower().map(response_mapping)
        df[col] = pd.to_numeric(df[col], errors='coerce')
        
        # Show sample conversion
        if col == eq_cols[0]:
            sample_vals = df[col].dropna().head(5).tolist()
            print(f"  Sample {col} after conversion: {sample_vals}")

print(f"\n✅ Converted {len(all_q_cols)} questionnaire columns to numeric")


## STEP 3: Score Questionnaires (Matching C4 Rules)

### EQ-10 Scoring (Binary 0-1 with reverse-scoring)
- Items 1,2,4,5,6,7,8,9,10: Agree (3,4) = 1 point
- Item 3: Disagree (1,2) = 1 point (reverse-scored)

### SQR-10 Scoring (Binary 0-1 with reverse-scoring)  
- Items 1,3,5,7,9: Agree (3,4) = 1 point
- Items 2,4,6,8,10: Disagree (1,2) = 1 point (reverse-scored)

### AQ-10 Scoring (For Reference Only - Will Be Excluded)
- Items 1,7,8,10: Agree (3,4) = 1 point
- Items 2,3,4,5,6,9: Disagree (1,2) = 1 point (reverse-scored)

### SPQ-10 (Missing in YBT - Fill with 0)
- YBT does NOT have SPQ - all SPQ features will be set to 0


In [ ]:
print("="*80)
print("SCORING QUESTIONNAIRES (MATCHING C4 RULES)")
print("="*80)

# EQ-10 Scoring (Binary 0-1 with reverse-scoring)
print("\nEQ-10 Scoring...")
eq_reverse_items = [3]  # Item 3 is reverse-scored

for i in range(1, 11):
    col_name = f'eq10_{i}'
    if col_name in df.columns:
        if i in eq_reverse_items:
            # Reverse: disagree (1,2) = 1, agree (3,4) = 0
            df[col_name] = df[col_name].apply(
                lambda x: 1 if pd.notna(x) and x in [1, 2] 
                else 0 if pd.notna(x) and x in [3, 4] 
                else np.nan
            )
        else:
            # Normal: agree (3,4) = 1, disagree (1,2) = 0
            df[col_name] = df[col_name].apply(
                lambda x: 1 if pd.notna(x) and x in [3, 4] 
                else 0 if pd.notna(x) and x in [1, 2] 
                else np.nan
            )

# Calculate EQ total
df['eq_total'] = df[eq_cols].sum(axis=1)

# Map eq10_* to eq_* for C4 compatibility
for i in range(1, 11):
    if f'eq10_{i}' in df.columns:
        df[f'eq_{i}'] = df[f'eq10_{i}']

print(f"  ✅ EQ-10 scored: total range {df['eq_total'].min():.0f}-{df['eq_total'].max():.0f}, mean {df['eq_total'].mean():.2f}")


In [ ]:
# SQR-10 Scoring (Binary 0-1 with reverse-scoring)
print("\nSQR-10 Scoring...")
sqr_reverse_items = [2, 4, 6, 8, 10]  # Items that need reverse scoring

for i in range(1, 11):
    col_name = f'sq10_{i}'
    if col_name in df.columns:
        if i in sqr_reverse_items:
            # Reverse: disagree (1,2) = 1, agree (3,4) = 0
            df[col_name] = df[col_name].apply(
                lambda x: 1 if pd.notna(x) and x in [1, 2] 
                else 0 if pd.notna(x) and x in [3, 4] 
                else np.nan
            )
        else:
            # Normal: agree (3,4) = 1, disagree (1,2) = 0
            df[col_name] = df[col_name].apply(
                lambda x: 1 if pd.notna(x) and x in [3, 4] 
                else 0 if pd.notna(x) and x in [1, 2] 
                else np.nan
            )

# Calculate SQR total
df['sqr_total'] = df[sqr_cols].sum(axis=1)

# Map sq10_* to sqr_* for C4 compatibility
for i in range(1, 11):
    if f'sq10_{i}' in df.columns:
        df[f'sqr_{i}'] = df[f'sq10_{i}']

print(f"  ✅ SQR-10 scored: total range {df['sqr_total'].min():.0f}-{df['sqr_total'].max():.0f}, mean {df['sqr_total'].mean():.2f}")


In [ ]:
# AQ-10 Scoring (For Reference Only - Will Be Excluded from Features)
print("\nAQ-10 Scoring (for reference only - will be excluded)...")
aq_reverse_items = [2, 3, 4, 5, 6, 9]  # Items that need reverse scoring

for i in range(1, 11):
    col_name = f'aq_{i}'
    if col_name in df.columns:
        if i in aq_reverse_items:
            df[col_name] = df[col_name].apply(
                lambda x: 1 if pd.notna(x) and x in [1, 2] 
                else 0 if pd.notna(x) and x in [3, 4] 
                else np.nan
            )
        else:
            df[col_name] = df[col_name].apply(
                lambda x: 1 if pd.notna(x) and x in [3, 4] 
                else 0 if pd.notna(x) and x in [1, 2] 
                else np.nan
            )

# Calculate AQ total (for reference only)
df['aq_total'] = df[aq_cols].sum(axis=1)

print(f"  ✅ AQ-10 scored: total range {df['aq_total'].min():.0f}-{df['aq_total'].max():.0f}, mean {df['aq_total'].mean():.2f}")
print(f"  ⚠️  NOTE: AQ features will be EXCLUDED from final feature set (data leakage prevention)")


In [ ]:
# SPQ-10 (Missing in YBT - Fill with 0)
print("\nSPQ-10 (Missing in YBT - Filling with 0)...")
for i in range(1, 11):
    df[f'spq_{i}'] = 0

df['spq_total'] = 0

print(f"  ✅ Created SPQ features (all set to 0): spq_1 through spq_10, spq_total")
print(f"  ✅ This is expected - YBT does not have SPQ (Sensory Perception Quotient)")

print("\n" + "="*80)
print("QUESTIONNAIRE SCORING COMPLETE")
print("="*80)


## STEP 4: Create Target Variable


In [ ]:
print("="*80)
print("CREATING TARGET VARIABLE")
print("="*80)

# Create autism_target from diagnosis columns
if 'diagnosis' in df.columns:
    # Check if 'autism' appears in diagnosis column (case-insensitive)
    df['autism_target'] = df['diagnosis'].astype(str).str.contains('autism', case=False, na=False).astype(int)
    
    # Also check diagnosis_yes_no if available
    if 'diagnosis_yes_no' in df.columns:
        # Clean diagnosis_yes_no
        diagnosis_yes = (df['diagnosis_yes_no'].astype(str).str.lower().str.strip() == 'yes').astype(int)
        # If diagnosis_yes_no=1 but no autism in diagnosis, check if it's a valid autism case
        # For now, use diagnosis column as primary source
        pass
    
    print(f"\nAutism target distribution:")
    print(df['autism_target'].value_counts().to_dict())
    print(f"Autism prevalence: {df['autism_target'].mean()*100:.2f}%")
else:
    print("\n⚠️  WARNING: No diagnosis column found - cannot create autism_target")
    print("   Available columns:", [c for c in df.columns if 'diagnos' in c.lower()])
    df['autism_target'] = 0


## STEP 4.5: Balance YBT Dataset (1:1 Matching C4 Training)

**CRITICAL**: C4 models were trained on balanced data (1:1 autism:non-autism ratio). 
YBT must be balanced to match this distribution for fair validation.

**Why this matters:**
- C4 training: 50% autism, 50% non-autism
- YBT raw: 3.13% autism, 96.87% non-autism  
- Models trained on balanced data perform poorly on imbalanced data
- Balancing ensures fair comparison and proper model evaluation

In [ ]:
print("="*80)
print("BALANCING YBT DATASET (1:1 MATCHING C4 TRAINING)")
print("="*80)

from sklearn.utils import resample

# Check current distribution
print(f"\nOriginal YBT distribution:")
print(df['autism_target'].value_counts().to_dict())
print(f"Autism prevalence: {df['autism_target'].mean()*100:.2f}%")
print(f"Imbalance ratio: {(df['autism_target'] == 0).sum() / (df['autism_target'] == 1).sum():.2f}:1")

# Separate classes
autism_cases = df[df['autism_target'] == 1].copy()
non_autism_cases = df[df['autism_target'] == 0].copy()

print(f"\nAutism cases: {len(autism_cases)}")
print(f"Non-autism cases: {len(non_autism_cases)}")

# Downsample non-autism to match autism count (1:1 balance)
n_autism = len(autism_cases)
non_autism_downsampled = resample(
    non_autism_cases, 
    replace=False,  # No replacement (sampling without replacement)
    n_samples=n_autism, 
    random_state=42
)

# Combine balanced datasets
df_balanced = pd.concat([autism_cases, non_autism_downsampled], ignore_index=True)

# Shuffle to randomize order
df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"\n✅ Balanced YBT dataset:")
print(f"   Total samples: {len(df_balanced)}")
print(f"   Autism cases: {(df_balanced['autism_target'] == 1).sum()}")
print(f"   Non-autism cases: {(df_balanced['autism_target'] == 0).sum()}")
print(f"   Balance ratio: 1:1")
print(f"   Autism prevalence: {df_balanced['autism_target'].mean()*100:.2f}%")

# Replace df with balanced version for subsequent steps
df = df_balanced.copy()

print(f"\n✅ Dataset balanced - proceeding with balanced YBT for validation")

## STEP 5: Demographic Feature Engineering


In [ ]:
print("="*80)
print("DEMOGRAPHIC FEATURE ENGINEERING")
print("="*80)

# Age: Use 'age' column, ensure numeric
df['age'] = pd.to_numeric(df['age'], errors='coerce')
age_median = df['age'].median()
df['age'] = df['age'].fillna(age_median)
print(f"\nAge: median={age_median:.1f}, range={df['age'].min():.0f}-{df['age'].max():.0f}")

# Sex: Map to numeric (matching C4 encoding)
# C4 uses: 1=male, 2=female, 3=other, 4=prefer not to say
sex_mapping = {
    'male': 1,
    'female': 2,
    'other': 3,
    'prefer not to say': 4,
    'i prefer not to say': 4,
    'i do not know': 4
}

if 'sex' in df.columns:
    df['sex'] = df['sex'].astype(str).str.strip().str.lower().map(sex_mapping).fillna(4)
    df['sex_num'] = df['sex'].map({1: 0, 2: 1, 3: 2, 4: 3}).fillna(0).astype(int)
    print(f"Sex: {df['sex'].value_counts().to_dict()}")
else:
    df['sex'] = 4  # Unknown
    df['sex_num'] = 0
    print("⚠️  Sex column not found - set to unknown (4)")

# Age groups (matching C4 bins)
df['age_group_19-30'] = ((df['age'] >= 19) & (df['age'] <= 30)).astype(int)
df['age_group_31-45'] = ((df['age'] >= 31) & (df['age'] <= 45)).astype(int)
df['age_group_46-60'] = ((df['age'] >= 46) & (df['age'] <= 60)).astype(int)
df['age_group_61+'] = (df['age'] >= 61).astype(int)

print(f"\nAge groups:")
print(f"  19-30: {df['age_group_19-30'].sum()}")
print(f"  31-45: {df['age_group_31-45'].sum()}")
print(f"  46-60: {df['age_group_46-60'].sum()}")
print(f"  61+: {df['age_group_61+'].sum()}")

# sqrt_age
df['sqrt_age'] = np.sqrt(np.clip(df['age'], a_min=0, a_max=None))
print(f"\n✅ sqrt_age created: range {df['sqrt_age'].min():.2f}-{df['sqrt_age'].max():.2f}")


## STEP 6: Questionnaire Totals and Interactions


In [ ]:
print("="*80)
print("QUESTIONNAIRE TOTALS AND INTERACTIONS")
print("="*80)

# d_score (SQR - EQ, matching C4)
df['d_score'] = df['sqr_total'] - df['eq_total']
print(f"\nd_score (SQR - EQ): range {df['d_score'].min():.0f}-{df['d_score'].max():.0f}, mean {df['d_score'].mean():.2f}")

# age_x_eq interaction
df['age_x_eq'] = df['age'] * df['eq_total']
print(f"age_x_eq: range {df['age_x_eq'].min():.0f}-{df['age_x_eq'].max():.0f}, mean {df['age_x_eq'].mean():.2f}")

# eq_sqr_ratio
df['eq_sqr_ratio'] = df['eq_total'] / (df['sqr_total'].replace(0, np.nan) + 1e-8)
df['eq_sqr_ratio'] = df['eq_sqr_ratio'].replace([np.inf, -np.inf], np.nan).fillna(0.0)
print(f"eq_sqr_ratio: range {df['eq_sqr_ratio'].min():.2f}-{df['eq_sqr_ratio'].max():.2f}, mean {df['eq_sqr_ratio'].mean():.2f}")

print("\n✅ All questionnaire interactions created")


## STEP 7: Occupation Feature


In [ ]:
print("="*80)
print("OCCUPATION FEATURE")
print("="*80)

# is_stem_occupation (binary flag)
if 'Q549' in df.columns or 'occupation' in df.columns:
    occupation_col = 'Q549' if 'Q549' in df.columns else 'occupation'
    df['is_stem_occupation'] = df[occupation_col].astype(str).str.contains(
        'science|technology|engineering|math|computer|software|data|research', 
        case=False, na=False
    ).astype(int)
    print(f"\n✅ is_stem_occupation created: {df['is_stem_occupation'].sum()} STEM occupations out of {len(df)}")
else:
    df['is_stem_occupation'] = 0
    print("\n⚠️  Occupation column not found - set is_stem_occupation to 0")


## STEP 8: Feature Alignment to C4 Schema

This is the critical step - align YBT features to exactly match C4's 45-feature schema in the correct order.


In [ ]:
print("="*80)
print("FEATURE ALIGNMENT TO C4 SCHEMA")
print("="*80)

# Load C4 feature schema
c4_feature_names = feature_info['feature_names']  # 45 features
excluded_features = set(feature_info.get('excluded_features', []))

print(f"\nC4 expects {len(c4_feature_names)} features")
print(f"Excluded features (AQ-related, will not be used): {len(excluded_features)}")
print(f"  Examples: {list(excluded_features)[:5]}")

# Build aligned feature matrix
X_ybt = pd.DataFrame(index=df.index)
missing_features = []
available_features = []

for feat_name in c4_feature_names:
    if feat_name in df.columns:
        X_ybt[feat_name] = df[feat_name]
        available_features.append(feat_name)
    else:
        # Missing feature - fill with 0
        X_ybt[feat_name] = 0
        missing_features.append(feat_name)

# Ensure correct order (CRITICAL)
X_ybt = X_ybt[c4_feature_names]

print(f"\n✅ Aligned feature matrix shape: {X_ybt.shape}")
print(f"✅ Available features: {len(available_features)}/{len(c4_feature_names)}")
print(f"⚠️  Missing features filled with 0: {len(missing_features)}")

if missing_features:
    print(f"\nMissing features (filled with 0):")
    for feat in missing_features[:10]:
        print(f"  - {feat}")
    if len(missing_features) > 10:
        print(f"  ... and {len(missing_features) - 10} more")

# Handle missing values and ensure numeric
X_ybt = X_ybt.fillna(0)
X_ybt = X_ybt.apply(pd.to_numeric, errors='coerce').fillna(0)

# Check for infinite values
inf_cols = []
for col in X_ybt.columns:
    if np.isinf(X_ybt[col]).any():
        inf_cols.append(col)
        X_ybt[col] = X_ybt[col].replace([np.inf, -np.inf], 0)

if inf_cols:
    print(f"\n⚠️  Found infinite values in: {inf_cols} (replaced with 0)")
else:
    print(f"\n✅ No infinite values found")

print(f"\n✅ Feature alignment complete - ready for scaling")


## STEP 9: Apply C4 Scaler (NO REFITTING)

**CRITICAL**: Use the scaler fitted on C4 training data. Do NOT refit on YBT.


In [ ]:
print("="*80)
print("APPLYING C4 SCALER (NO REFITTING)")
print("="*80)

# Apply scaler (fitted on C4, NOT refit on YBT)
X_ybt_scaled = scaler.transform(X_ybt.values)

print(f"\n✅ Scaled feature matrix shape: {X_ybt_scaled.shape}")
print(f"✅ Applied C4 scaler (fitted on C4 training data, NOT refit on YBT)")
print(f"✅ Feature order matches C4 exactly: {list(X_ybt.columns[:5])}...")

# Verify scaling worked
print(f"\nScaled feature statistics:")
print(f"  Mean: {X_ybt_scaled.mean():.4f}")
print(f"  Std: {X_ybt_scaled.std():.4f}")
print(f"  Min: {X_ybt_scaled.min():.4f}")
print(f"  Max: {X_ybt_scaled.max():.4f}")


## STEP 10: Generate Predictions from All Models


In [ ]:
print("="*80)
print("GENERATING PREDICTIONS FROM ALL MODELS")
print("="*80)

# Generate predictions
predictions = {}
probabilities = {}

for name, model in loaded_models.items():
    y_proba = model.predict_proba(X_ybt_scaled)[:, 1]
    y_pred = (y_proba >= 0.5).astype(int)
    predictions[name] = y_pred
    probabilities[name] = y_proba
    print(f"\n{name}:")
    print(f"  Generated predictions for {len(y_pred)} samples")
    print(f"  Probability range: {y_proba.min():.4f} - {y_proba.max():.4f}")
    print(f"  Predicted positives: {y_pred.sum()} ({y_pred.mean()*100:.1f}%)")

print("\n✅ Predictions generated from all models")


## STEP 11: Evaluate Performance (Balanced YBT Dataset)

**Note**: Evaluation is performed on the balanced YBT dataset (1:1 ratio) to match the C4 training distribution.
This ensures fair comparison and proper assessment of model discriminative power.


In [ ]:
print("="*80)
print("EVALUATING PERFORMANCE")
print("="*80)

metrics_df = None

if 'autism_target' in df.columns:
    y_true = df['autism_target'].values
    
    print(f"\nGround truth available:")
    print(f"  Total samples: {len(y_true)}")
    print(f"  Autism cases: {y_true.sum()} ({y_true.mean()*100:.2f}%)")
    print(f"  Non-autism cases: {(y_true == 0).sum()} ({(y_true == 0).mean()*100:.2f}%)")
    
    results = {}
    for name in loaded_models.keys():
        y_pred = predictions[name]
        y_proba = probabilities[name]
        
        results[name] = {
            'accuracy': accuracy_score(y_true, y_pred),
            'precision': precision_score(y_true, y_pred, zero_division=0),
            'recall': recall_score(y_true, y_pred, zero_division=0),
            'f1': f1_score(y_true, y_pred, zero_division=0),
            'auc': roc_auc_score(y_true, y_proba)
        }
    
    metrics_df = pd.DataFrame(results).T
    
    print("\n" + "="*80)
    print("EXTERNAL VALIDATION RESULTS ON YBT")
    print("="*80)
    print(metrics_df.round(4).sort_values('auc', ascending=False))
    
    # Compare with C4 performance
    print("\n" + "="*80)
    print("COMPARISON WITH C4 PERFORMANCE")
    print("="*80)
    
    # Load C4 results for comparison
    c4_results_path = os.path.join(ARTIFACT_DIR, 'original_dataset_results.json')
    if os.path.exists(c4_results_path):
        with open(c4_results_path, 'r') as f:
            c4_results = json.load(f)
        
        comparison_data = []
        for model_name in metrics_df.index:
            if model_name in c4_results:
                comparison_data.append({
                    'Model': model_name,
                    'C4_F1': c4_results[model_name]['f1'],
                    'YBT_F1': metrics_df.loc[model_name, 'f1'],
                    'C4_AUC': c4_results[model_name]['auc'],
                    'YBT_AUC': metrics_df.loc[model_name, 'auc'],
                    'F1_Drop': c4_results[model_name]['f1'] - metrics_df.loc[model_name, 'f1'],
                    'AUC_Drop': c4_results[model_name]['auc'] - metrics_df.loc[model_name, 'auc']
                })
        
        comparison_df = pd.DataFrame(comparison_data)
        print(comparison_df.round(4))
    
    # Save results
    output_dir = '/Users/eb2007/playground/bullpy/c4_play2/data/processed'
    os.makedirs(output_dir, exist_ok=True)
    metrics_df.to_csv(os.path.join(output_dir, 'ybt_external_validation_results.csv'))
    print(f"\n✅ Results saved to: {output_dir}/ybt_external_validation_results.csv")
    
else:
    print("\n⚠️  No ground truth labels available - skipping evaluation")
    print("   Predictions will be saved but metrics cannot be calculated")


## STEP 12: Save Predictions and Feature Alignment Info


In [ ]:
print("="*80)
print("SAVING PREDICTIONS AND METADATA")
print("="*80)

output_dir = '/Users/eb2007/playground/bullpy/c4_play2/data/processed'
os.makedirs(output_dir, exist_ok=True)

# Save predictions
pred_df = pd.DataFrame({
    'userid': df.index if 'userid' not in df.columns else df['userid'].values
})

for name in loaded_models.keys():
    pred_df[f'proba_{name.replace(" ", "_").lower()}'] = probabilities[name]
    pred_df[f'pred_{name.replace(" ", "_").lower()}'] = predictions[name]

# Add ground truth if available
if 'autism_target' in df.columns:
    pred_df['autism_target'] = df['autism_target'].values

pred_path = os.path.join(output_dir, 'ybt_external_predictions.csv')
pred_df.to_csv(pred_path, index=False)
print(f"\n✅ Predictions saved to: {pred_path}")

# Save feature alignment info
alignment_info = {
    'c4_feature_count': len(c4_feature_names),
    'ybt_available_features': len(available_features),
    'ybt_missing_features': len(missing_features),
    'missing_features': missing_features,
    'available_features': available_features,
    'excluded_features': list(excluded_features),
    'note': 'SPQ features filled with 0 (YBT does not have SPQ). AQ features excluded to prevent data leakage.'
}

alignment_path = os.path.join(output_dir, 'ybt_feature_alignment.json')
with open(alignment_path, 'w') as f:
    json.dump(alignment_info, f, indent=2)
print(f"✅ Feature alignment info saved to: {alignment_path}")

print("\n" + "="*80)
print("EXTERNAL VALIDATION COMPLETE")
print("="*80)
print(f"\nSummary:")
print(f"  Dataset: YBT")
print(f"  Samples: {len(df)}")
print(f"  Features: {X_ybt_scaled.shape[1]} (aligned to C4)")
print(f"  Models tested: {len(loaded_models)}")
if metrics_df is not None:
    best_model = metrics_df['auc'].idxmax()
    print(f"  Best model: {best_model} (AUC: {metrics_df.loc[best_model, 'auc']:.4f})")
print(f"\n✅ All outputs saved to: {output_dir}")


# External Validation on CARD Dataset

This notebook loads the saved C4 training artifacts (feature schema, scaler, models), prepares the CARD dataset to match the training feature space, applies the trained models, and reports performance and outputs predictions.

- Leakage-free: uses the saved scaler (fitted on C4 train) and does not refit on CARD
- Strict feature alignment: uses the exact feature names and order from training
- Robust preprocessing: replicates questionnaire scoring and key feature engineering used in C4



In [ ]:
# Imports and config
import os
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score, roc_curve

# Paths
ARTIFACT_DIR = '/Users/eb2007/playground/bullpy/c4_play2/models/cross_validation'
FEATURE_INFO_PATH = os.path.join(ARTIFACT_DIR, 'feature_info_original.json')
SCALER_PATH = os.path.join(ARTIFACT_DIR, 'scaler_original.joblib')
MODELS = {
    'Logistic Regression': os.path.join(ARTIFACT_DIR, 'logistic_regression_original.joblib'),
    'Random Forest': os.path.join(ARTIFACT_DIR, 'random_forest_original.joblib'),
    'XGBoost': os.path.join(ARTIFACT_DIR, 'xgboost_original.joblib'),
    'LightGBM': os.path.join(ARTIFACT_DIR, 'lightgbm_original.joblib'),
    'Gradient Boosting': os.path.join(ARTIFACT_DIR, 'gradient_boosting_original.joblib'),
}

# CARD dataset path
DATA_PATH = '/Users/eb2007/Library/CloudStorage/OneDrive-UniversityofCambridge/Documents/PhD/data/CARD_Noc2025.xlsx'

np.random.seed(42)



In [ ]:
# Load training artifacts (feature schema, scaler, models)
with open(FEATURE_INFO_PATH, 'r') as f:
    feature_info = json.load(f)

feature_names = feature_info['feature_names']
excluded_features = set(feature_info.get('excluded_features', []))
print(f"Loaded feature schema with {len(feature_names)} features")

scaler = joblib.load(SCALER_PATH)
print("Loaded saved StandardScaler (trained on C4)")

loaded_models = {}
for name, path in MODELS.items():
    if os.path.exists(path):
        loaded_models[name] = joblib.load(path)
print(f"Loaded {len(loaded_models)} trained models: {list(loaded_models.keys())}")



In [ ]:
# Load CARD dataset
print(f"Loading CARD data from: {DATA_PATH}")
if DATA_PATH.lower().endswith(('.xlsx', '.xls')):
    df_card = pd.read_excel(DATA_PATH)
else:
    df_card = pd.read_csv(DATA_PATH)
print(df_card.shape)
print("Columns:", list(df_card.columns)[:30], '...')

# Basic cleaning
if 'userid' in df_card.columns:
    df_card = df_card.drop_duplicates(subset=['userid'])
else:
    df_card = df_card.drop_duplicates()

# Ensure expected types are compatible
if 'occupation' in df_card.columns:
    df_card['occupation'] = df_card['occupation'].astype(str)

# Coerce numeric-looking columns
for col in df_card.columns:
    if df_card[col].dtype == 'object':
        try:
            df_card[col] = pd.to_numeric(df_card[col])
        except Exception:
            pass

print("After basic cleaning:", df_card.shape)



In [ ]:
# Questionnaire scoring (replicate C4 rules)
# SPQ-10 items: 1->3, 2->2, 3->1, 4->0; total 0-30
spq_cols = [c for c in df_card.columns if c.lower().startswith('spq_')]
for c in spq_cols:
    df_card[c] = df_card[c].map({1: 3, 2: 2, 3: 1, 4: 0})
if spq_cols:
    df_card['spq_total'] = df_card[spq_cols].sum(axis=1)

# EQ-10 items: 1->1, 2/3/4->0; total 0-10
eq_cols = [c for c in df_card.columns if c.lower().startswith('eq_')]
for c in eq_cols:
    df_card[c] = df_card[c].map({1: 1, 2: 0, 3: 0, 4: 0})
if eq_cols:
    df_card['eq_total'] = df_card[eq_cols].sum(axis=1)

# SQR-10 items: 1->1, 2/3/4->0; total 0-10
sqr_cols = [c for c in df_card.columns if c.lower().startswith('sqr_')]
for c in sqr_cols:
    df_card[c] = df_card[c].map({1: 1, 2: 0, 3: 0, 4: 0})
if sqr_cols:
    df_card['sqr_total'] = df_card[sqr_cols].sum(axis=1)

# AQ-10 items: 1->1, 2/3/4->0; total 0-10
aq_cols = [c for c in df_card.columns if c.lower().startswith('aq_')]
for c in aq_cols:
    df_card[c] = df_card[c].map({1: 1, 2: 0, 3: 0, 4: 0})
if aq_cols:
    df_card['aq_total'] = df_card[aq_cols].sum(axis=1)

print('Scoring complete:')
print({
    'spq_items': len(spq_cols),
    'eq_items': len(eq_cols),
    'sqr_items': len(sqr_cols),
    'aq_items': len(aq_cols)
})



In [ ]:
# Feature engineering to match training
# Age groups and transforms
if 'age' in df_card.columns:
    df_card['sqrt_age'] = np.sqrt(np.clip(df_card['age'], a_min=0, a_max=None))
    df_card['age_group_19-30'] = ((df_card['age'] >= 19) & (df_card['age'] <= 30)).astype(int)
    df_card['age_group_31-45'] = ((df_card['age'] >= 31) & (df_card['age'] <= 45)).astype(int)
    df_card['age_group_46-60'] = ((df_card['age'] >= 46) & (df_card['age'] <= 60)).astype(int)
    df_card['age_group_61+'] = (df_card['age'] >= 61).astype(int)
else:
    df_card['sqrt_age'] = 0.0
    df_card['age_group_19-30'] = 0
    df_card['age_group_31-45'] = 0
    df_card['age_group_46-60'] = 0
    df_card['age_group_61+'] = 0

# sex_num mapping as used in C4 (fallback to 0 if missing)
if 'sex' in df_card.columns:
    df_card['sex_num'] = df_card['sex'].map({1: 0, 2: 1, 3: 2, 4: 3}).fillna(0).astype(int)
else:
    df_card['sex_num'] = 0

# STEM occupation
if 'occupation' in df_card.columns:
    df_card['is_stem_occupation'] = df_card['occupation'].str.contains(
        'science|technology|engineering|math|computer|software|data|research', case=False, na=False
    ).astype(int)
else:
    df_card['is_stem_occupation'] = 0

# Interactions and ratios consistent with feature_info
if {'age', 'eq_total'}.issubset(df_card.columns):
    df_card['age_x_eq'] = df_card['age'] * df_card['eq_total']
else:
    df_card['age_x_eq'] = 0.0

if {'eq_total', 'sqr_total'}.issubset(df_card.columns):
    df_card['eq_sqr_ratio'] = df_card['eq_total'] / (df_card['sqr_total'].replace(0, np.nan))
    df_card['eq_sqr_ratio'] = df_card['eq_sqr_ratio'].replace([np.inf, -np.inf], np.nan).fillna(0.0)
else:
    df_card['eq_sqr_ratio'] = 0.0

# d_score (difference between SQR and EQ; sign consistent with prior code)
if {'sqr_total', 'eq_total'}.issubset(df_card.columns):
    df_card['d_score'] = df_card['sqr_total'] - df_card['eq_total']
else:
    df_card['d_score'] = 0.0



In [ ]:
# Build aligned feature matrix in exact training order
X_card = pd.DataFrame(index=df_card.index)
missing_from_card = []
for fname in feature_names:
    if fname in df_card.columns:
        X_card[fname] = df_card[fname]
    else:
        # Create missing feature as 0
        X_card[fname] = 0
        missing_from_card.append(fname)

# Basic type coercion and missing handling
for c in X_card.columns:
    if X_card[c].dtype == 'object':
        X_card[c] = pd.Categorical(X_card[c]).codes

X_card = X_card.apply(pd.to_numeric, errors='coerce')
num_missing = int(X_card.isnull().sum().sum())
if num_missing > 0:
    X_card = X_card.fillna(X_card.median(numeric_only=True))

print(f"Aligned feature matrix shape: {X_card.shape}")
if missing_from_card:
    print(f"Note: Missing {len(missing_from_card)} features in CARD, filled with 0: {missing_from_card[:10]}...")



In [ ]:
# Apply saved scaler (no refit)
X_card_scaled = scaler.transform(X_card.values)

# Predict with each model
results = {}
probas = {}
for name, model in loaded_models.items():
    y_proba = model.predict_proba(X_card_scaled)[:, 1]
    y_pred = (y_proba >= 0.5).astype(int)
    probas[name] = y_proba
    results[name] = {'n': len(y_pred)}

print('Predictions generated for models:', list(results.keys()))



In [ ]:
# Evaluate if ground-truth label present
metrics_df = None
label_col_candidates = ['autism_target', 'diagnosis_autism', 'has_autism']
label_col = next((c for c in label_col_candidates if c in df_card.columns), None)

if label_col is not None:
    y_true = df_card[label_col].astype(int).clip(0, 1).values
    for name, model in loaded_models.items():
        y_proba = probas[name]
        y_pred = (y_proba >= 0.5).astype(int)
        results[name].update({
            'accuracy': accuracy_score(y_true, y_pred),
            'precision': precision_score(y_true, y_pred, zero_division=0),
            'recall': recall_score(y_true, y_pred, zero_division=0),
            'f1': f1_score(y_true, y_pred, zero_division=0),
            'auc': roc_auc_score(y_true, y_proba)
        })
    metrics_df = pd.DataFrame(results).T
    print('External validation metrics (CARD):')
    print(metrics_df.round(4).sort_values('auc', ascending=False))
else:
    print('No label column found; skipping metrics. Saving predictions only.')



In [ ]:
# Save outputs
os.makedirs('/Users/eb2007/playground/bullpy/c4_play2/data/processed', exist_ok=True)

pred_df = pd.DataFrame({'userid': df_card['userid'] if 'userid' in df_card.columns else np.arange(len(df_card))})
for name, y_proba in probas.items():
    pred_df[f'proba_{name.replace(" ", "_").lower()}'] = y_proba

pred_path = '/Users/eb2007/playground/bullpy/c4_play2/data/processed/card_external_predictions.csv'
pred_df.to_csv(pred_path, index=False)
print(f"Predictions saved to: {pred_path}")

feat_used_path = '/Users/eb2007/playground/bullpy/c4_play2/data/processed/card_features_used.json'
with open(feat_used_path, 'w') as f:
    json.dump({'feature_names': feature_names, 'missing_filled_zero': missing_from_card}, f, indent=2)
print(f"Feature alignment info saved to: {feat_used_path}")

if metrics_df is not None:
    metrics_path = '/Users/eb2007/playground/bullpy/c4_play2/data/processed/card_external_metrics.csv'
    metrics_df.to_csv(metrics_path)
    print(f"Metrics saved to: {metrics_path}")

